In [109]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import LabelEncoder,OneHotEncoder,MinMaxScaler,RobustScaler,label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score,f1_score,recall_score,precision_score,confusion_matrix,roc_auc_score,balanced_accuracy_score

N_SPLITS = 3
RANDOM_STATE=42
skf = StratifiedKFold(n_splits=N_SPLITS,shuffle=True,random_state=RANDOM_STATE)

data_path = Path.cwd().parent / "data" / "playground-series-s6e6"
data_orig_path = Path.cwd().parent / "data" / "star_classification.csv"

df_train = pd.read_csv(data_path / "train.csv")
df_test = pd.read_csv(data_path / "test.csv")
data_orig = pd.read_csv(data_orig_path)

df_exp = df_train.copy()
df_test_exp = df_test.copy()

train_ids = df_exp["id"].values
test_ids = df_test_exp["id"].values

df_exp.drop(["id"],axis=1,inplace=True)
X_test = df_test_exp.drop(["id"],axis=1)

X_train,y_train = df_exp.drop(["class"],axis=1), df_exp["class"]

target_col = ["class"]
categorical_cols = ["spectral_type","galaxy_population"]
numerical_cols = [col for col in X_train.columns if col not in categorical_cols + target_col ]
objects_for_test = {}

In [110]:

def create_astronomical_bins(df:pd.Dataframe):
    if "spectral_type" not in df.columns:
        df["spectral_type"] = pd.cut(df["r"] - df["g"],bins=[-np.inf,-1,-0.5,0,np.inf],labels=["M","G/K","A/F","O/B"]).astype("str")
    if "galaxy_population" not in df.columns:
        df["galaxy_population"] = pd.cut(df["u"] - df["r"],bins=[-np.inf,2.2,np.inf],labels=["Blue_Cloud","Red_Sequence"]).astype("str")
    if "combo" not in df.columns:
        df["combo"] = df["spectral_type"] + "_" + df["galaxy_population"]

    return df

def generate_target_encodings(train_df:pd.DataFrame,test_df:pd.DataFrame,orig_df:pd.DataFrame):
    orig_df = create_astronomical_bins(orig_df)
    target_map = {"GALAXY":0,"QSO":1,"STAR":2}
    y_orig = orig_df["class"].map(target_map)

    spectral_map = y_orig.groupby(orig_df["spectral_type"]).mean()
    galaxypop_map = y_orig.groupby(orig_df["galaxy_population"]).mean()
    combo_map = y_orig.groupby(orig_df["combo"]).mean()


    for df in [train_df,test_df]:
        df = create_astronomical_bins(df)
        df["orig_spectral_target"] = df["spectral_type"].map(spectral_map) 
        df["orig_galaxypop_target"] =df["galaxy_population"].map(galaxypop_map)
        df["orig_combo_target"] = df["combo"].map(combo_map)

    return train_df,test_df



In [111]:
X_train,X_test = generate_target_encodings(train_df=X_train,test_df=X_test,orig_df=data_orig)

In [ ]:
def feature_engineering(df:pd.DataFrame):
    bands = ["u","g","r","i","z"]
    C = 300000
    HC = 70
    epsilon = 1e-5
    abs_redshift = df['redshift'].abs() + epsilon
    redshift_factor = 5*np.log10(1+abs_redshift)

    print(type(redshift_factor))
    print(redshift_factor.shape)
    

    for i in bands:
        df[f"absolute_{i}"] = df[i]  - redshift_factor
    
    df["mag_mean"] = df[bands].mean(axis=1)
    df["mag_std"] = df[bands].std(axis=1)
    df["mag_max"] = df[bands].max(axis=1)
    df["mag_min"] = df[bands].min(axis=1)
    df["mag_range"] = df["mag_max"] - df["mag_min"]

    # color indices
    df['u_g'] = df['u'] - df['g']
    df['g_r'] = df['g'] - df['r']
    df['r_i'] = df['r'] - df['i']
    df['i_z'] = df['i'] - df['z']
    df['u_r'] = df['u'] - df['r']
    df['u_i'] = df['u'] - df['i']
    df['u_z'] = df['u'] - df['z']
    df['g_i'] = df['g'] - df['i']
    df['g_z'] = df['g'] - df['z']
    df['r_z'] = df['r'] - df['z']

    # flux conversion
    for b in bands:
        clipped_band = df[b].clip(lower=0,upper=35)
        df[f"flux_{b}"] = 10**(-0.4*clipped_band)
    
    flux_cols = [f'flux_{b}' for b in bands]
    df['flux_mean'] = df[flux_cols].mean(axis=1)
    df['flux_std'] = df[flux_cols].std(axis=1)
    df['flux_max'] = df[flux_cols].max(axis=1)
    df['flux_min'] = df[flux_cols].min(axis=1)
    df['flux_range'] = df['flux_max'] - df['flux_min']

    alpha_rad = np.radians(df['alpha'])
    delta_rad = np.radians(df['delta'])
    df['alpha_sin'] = np.sin(alpha_rad)
    df['alpha_cos'] = np.cos(alpha_rad)
    df['delta_sin'] = np.sin(delta_rad)
    df['delta_cos'] = np.cos(delta_rad)

 
    for col in ['u_g', 'g_r', 'r_i', 'i_z']:
        df[f'{col}_redshift'] = df[col] * redshift_factor
        
    return df


X_train=feature_engineering(X_train)
X_test=feature_engineering(X_test)

del data_orig



<class 'pandas.Series'>
(577347,)
<class 'pandas.Series'>
(247435,)


In [114]:
X_train.describe()

,alpha,delta,u,g,r,i,z,redshift,orig_spectral_target,orig_galaxypop_target,...,flux_min,flux_range,alpha_sin,alpha_cos,delta_sin,delta_cos,u_g_redshift,g_r_redshift,r_i_redshift,i_z_redshift
count,577347.000000,577347.000000,577347.000000,577347.000000,577347.000000,577347.000000,577347.000000,577347.000000,577347.000000,577347.000000,...,5.773470e+05,5.773470e+05,577347.000000,577347.000000,577347.000000,577347.000000,577347.000000,577347.000000,577347.000000,577347.000000
mean,181.616673,21.834654,22.441926,21.007273,19.962811,19.378911,19.041136,0.723135,0.585963,0.606705,...,5.235490e-09,4.982648e-06,-0.037899,-0.242639,0.351020,0.878817,1.208004,0.911386,0.544237,0.335269
std,96.242941,18.933570,2.018135,1.795426,1.648964,1.580059,1.584365,0.810070,0.387362,0.268085,...,1.518056e-08,2.156953e-03,0.565397,0.787412,0.293870,0.134561,1.591932,0.945667,0.659757,0.560937
min,0.011684,-17.966988,-0.139225,13.535483,12.579407,11.962781,11.682803,-0.009970,0.257720,0.365926,...,4.996831e-12,1.194951e-10,-0.999978,-1.000000,-0.308469,0.188096,-84.584780,-14.003444,-9.694118,-14.203627
25%,132.161499,2.474097,20.977090,19.865005,18.820671,18.306820,17.973192,0.181052,0.257720,0.365926,...,2.745208e-10,6.670005e-09,-0.579565,-0.853478,0.043168,0.798758,0.208990,0.149862,0.058695,0.018794
50%,188.681465,21.484412,22.570222,21.467820,20.431153,19.631642,19.188598,0.497525,0.257720,0.365926,...,9.006846e-10,2.046052e-08,-0.040506,-0.648506,0.366248,0.930517,0.789647,0.732816,0.404470,0.217317
75%,231.829693,36.988310,23.869103,22.292715,21.164096,20.608191,20.162111,0.881390,1.131426,0.905193,...,3.928227e-09,6.260802e-08,0.493426,0.818713,0.601652,0.997362,1.734035,1.486915,0.927254,0.575644
max,359.999810,79.158322,28.253263,27.620208,25.254499,27.910853,26.826867,7.010780,1.163880,0.905193,...,1.385255e-06,1.000000e+00,1.000000,1.000000,0.982151,1.000000,22.937956,19.297735,19.501325,27.332456


In [115]:
le = LabelEncoder()
y_train= le.fit_transform(y_train)

In [116]:
cat_cols = X_test.select_dtypes(["str"]).columns.to_list()
for col in cat_cols:
    X_train[col] = X_train[col].astype("category")
    X_test[col] =X_test[col].astype("category")

In [117]:
y_train = pd.Series(y_train)

In [ ]:
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb

xgb_params = {
    'objective': 'multi:softprob',
    'num_class': 3,
    'eval_metric': 'mlogloss',
    'tree_method': 'gpu_hist',           
    'learning_rate': 0.015,
    'n_estimators': 6000,
    'max_depth': 8,
    'min_child_weight': 5,
    'max_delta_step': 1,
    'gamma': 0.2,
    'reg_alpha': 0.5,
    'reg_lambda': 2.5,
    'subsample': 0.75,
    'colsample_bytree': 0.7,
    'colsample_bylevel': 0.8,
    'enable_categorical': True,
    'device':"cuda",
    'max_bins':1000
}

# Multi-seed configuration
SEEDS = [42]
n_splits = 5

print("Initiating Multi-Seed Cross-Validation sequence...")

for seed in SEEDS:
    print(f"\n--- Running Seed {seed} ---")
    xgb_params['random_state'] = seed
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    
    seed_scores = []
    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)
    
    model = xgb.XGBClassifier(**xgb_params)
    model.fit(
        X_train, y_train,
        sample_weight=sample_weights, 
        verbose=1000
    )
    
    preds = model.predict(X_test)
    
preds



Initiating Multi-Seed Cross-Validation sequence...

--- Running Seed 42 ---


array([0, 0, 0, ..., 0, 1, 0], shape=(247435,))

In [123]:
labels = le.inverse_transform(preds)
ids = df_test["id"].values
submission = pd.DataFrame({"id":ids,"class":labels})
submission.to_csv(Path.cwd().parent / "submissions" / "submission_v2_XGB.csv",index=False)
submission

,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY
...,...,...
247430,824777,QSO
247431,824778,QSO
247432,824779,GALAXY
247433,824780,QSO


In [ ]:
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb

xgb_params = {
    'objective': 'multi:softprob',
    'num_class': 3,
    'eval_metric': 'mlogloss',
    'tree_method': 'hist',           
    'learning_rate': 0.015,
    'n_estimators': 4000,
    'early_stopping_rounds': 150,
    'max_depth': 8,
    'min_child_weight': 5,
    'max_delta_step': 1,
    'gamma': 0.2,
    'reg_alpha': 0.5,
    'reg_lambda': 2.5,
    'subsample': 0.75,
    'colsample_bytree': 0.7,
    'colsample_bylevel': 0.8,
    'enable_categorical': True
}

# Multi-seed configuration
SEEDS = [42,2024,777]
n_splits = 5

oof_preds = np.zeros((len(X_train), 3))
test_preds = np.zeros((len(X_test), 3))

print("Initiating Multi-Seed Cross-Validation sequence...")

for seed in SEEDS:
    print(f"\n--- Running Seed {seed} ---")
    xgb_params['random_state'] = seed
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    
    seed_scores = []
    
    for fold, (train_idx, valid_idx) in enumerate(skf.split(X_train, y_train)):
        X, y = X_train.iloc[train_idx], y_train.iloc[train_idx]
        X_val, y_val = X_train.iloc[valid_idx], y_train.iloc[valid_idx]
        
        sample_weights = compute_sample_weight(class_weight='balanced', y=y)
        
        model = xgb.XGBClassifier(**xgb_params)
        model.fit(
            X, y,
            sample_weight=sample_weights, 
            eval_set=[(X, y), (X_val, y_val)],
            verbose=1000
        )
        
        valid_probs = model.predict_proba(X_val)
        
        # Aggregate OOF predictions
        oof_preds[valid_idx] += valid_probs / len(SEEDS)
        
        valid_class_preds = np.argmax(valid_probs, axis=1)
        score = balanced_accuracy_score(y_val, valid_class_preds)
        seed_scores.append(score)
        print(f"  Fold {fold+1} Score: {score:.5f}")
        
        # Aggregate Test predictions
        test_preds += model.predict_proba(X_test) / (n_splits * len(SEEDS))

    print(f"Seed {seed} Mean BA: {np.mean(seed_scores):.5f}")

# Calculate final blended metric
final_oof_classes = np.argmax(oof_preds, axis=1)
blended_score = balanced_accuracy_score(y, final_oof_classes)
print(f"\nFinal Blended OOF Balanced Accuracy: {blended_score:.5f}")



Initiating Multi-Seed Cross-Validation sequence...

--- Running Seed 42 ---
[0]	validation_0-mlogloss:1.08287	validation_1-mlogloss:1.08288
[1000]	validation_0-mlogloss:0.09305	validation_1-mlogloss:0.10748
[2000]	validation_0-mlogloss:0.07329	validation_1-mlogloss:0.10048
[3000]	validation_0-mlogloss:0.05949	validation_1-mlogloss:0.09717
[4000]	validation_0-mlogloss:0.04891	validation_1-mlogloss:0.09539
[5000]	validation_0-mlogloss:0.04063	validation_1-mlogloss:0.09459
[5999]	validation_0-mlogloss:0.03409	validation_1-mlogloss:0.09446
  Fold 1 Score: 0.96455
[0]	validation_0-mlogloss:1.08285	validation_1-mlogloss:1.08291
[1000]	validation_0-mlogloss:0.09324	validation_1-mlogloss:0.11032
[2000]	validation_0-mlogloss:0.07338	validation_1-mlogloss:0.10310
[3000]	validation_0-mlogloss:0.05977	validation_1-mlogloss:0.09986
[4000]	validation_0-mlogloss:0.04920	validation_1-mlogloss:0.09803
[5000]	validation_0-mlogloss:0.04085	validation_1-mlogloss:0.09721
[5999]	validation_0-mlogloss:0.0343

ValueError: Found input variables with inconsistent numbers of samples: [461878, 577347, 461878]